# Sentiment Analysis of Amazon Kitchen Reviews

**Dataset:** [Amazon Customer Reviews — US Kitchen Products](https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Kitchen_v1_00.tsv.gz)

---

## 1. Introduction

This notebook trains and evaluates four classical ML classifiers for binary sentiment classification on Amazon Kitchen product reviews. The pipeline moves from raw TSV data through text cleaning, preprocessing, TF-IDF feature extraction, and model training, with analysis at each stage.

**Research questions:**
1. Which classifier best generalises to unseen review text with TF-IDF features?
2. Does adding bigram features improve classification performance?
3. What vocabulary most strongly differentiates positive from negative reviews?
4. Where do models systematically fail?

**Sentiment label mapping:**
- ⭐⭐⭐⭐⭐ / ⭐⭐⭐⭐ (4–5 stars) → **Positive (1)**
- ⭐⭐ / ⭐ (1–2 stars) → **Negative (0)**
- ⭐⭐⭐ (3 stars) → **Discarded** (genuinely ambiguous)

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy nltk scikit-learn bs4 contractions wordcloud matplotlib

import re
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from bs4 import BeautifulSoup
import contractions
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
warnings.filterwarnings('ignore', category=UserWarning, module='bs4')

# Fixed seed — all results are fully reproducible
RANDOM_SEED = 544
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_colwidth', None)
print('Libraries loaded.')

---
## 2. Dataset Description

The dataset contains Amazon product reviews for US Kitchen products. Each row includes a star rating (1–5) and a free-text review body. Before any sampling, we inspect the raw class distribution to understand the natural imbalance in the data.

In [ ]:
fname = 'amazon_reviews_us_Kitchen_v1_00.tsv.gz'
original_data = pd.read_csv(fname, sep='\t', compression='gzip', on_bad_lines='skip')

data = original_data[['star_rating', 'review_body']].copy()
data.dropna(inplace=True)

print(f'Total reviews: {len(data):,}')
print(f"Mean rating:   {data.star_rating.mean():.3f} (std: {data.star_rating.std():.3f})")
print()

# Per-rating breakdown to make natural imbalance visible
for r in range(1, 6):
    n = len(data[data['star_rating'] == r])
    print(f'  Rating {r}: {n:>8,}  ({100 * n / len(data):.1f}%)')

---
## 3. Data Exploration

Before preprocessing, we look at sample reviews and raw length statistics to understand what we are working with.

In [ ]:
# Sample reviews from each extreme
print('=== Sample 5-star reviews ===')
for text in data[data['star_rating'] == 5]['review_body'].sample(2, random_state=RANDOM_SEED).values:
    print(f'  • {str(text)[:200]}')
    print()

print('=== Sample 1-star reviews ===')
for text in data[data['star_rating'] == 1]['review_body'].sample(2, random_state=RANDOM_SEED).values:
    print(f'  • {str(text)[:200]}')
    print()

In [ ]:
# Raw review length distribution before any processing
raw_lengths = data['review_body'].apply(lambda x: len(str(x)))

print(f'Review length stats (characters):')
print(f'  Mean:   {raw_lengths.mean():.0f}')
print(f'  Median: {raw_lengths.median():.0f}')
print(f'  Min:    {raw_lengths.min()}')
print(f'  Max:    {raw_lengths.max():,}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(raw_lengths.clip(upper=1500), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(raw_lengths.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {raw_lengths.mean():.0f}')
ax.set_title('Raw Review Length Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Character Length')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Data Cleaning

### Labelling and Balancing

Reviews rated 1–2 are labelled negative (0); 4–5 are labelled positive (1). Three-star reviews are discarded — they carry genuinely ambiguous sentiment and assigning them a binary label would introduce label noise.

100,000 reviews are sampled from each class to produce a balanced corpus. Training on the natural distribution (which heavily favours positive ratings) would bias the model toward predicting positive, inflating accuracy while degrading recall on negative examples.

In [ ]:
count_positive = len(data[data['star_rating'].isin([4, 5])])
count_negative = len(data[data['star_rating'].isin([1, 2])])
count_neutral  = len(data[data['star_rating'].isin([3])])
print(f'Positive: {count_positive:,} | Negative: {count_negative:,} | Neutral (discarded): {count_neutral:,}')

# Drop neutral and remap to binary labels
data.drop(data[data['star_rating'] == 3].index, inplace=True)
data['star_rating'].replace({1: 0, 2: 0, 4: 1, 5: 1}, inplace=True)
data['star_rating'] = data['star_rating'].astype('int8')

# Balanced sample
positive_reviews = data[data['star_rating'] == 1].sample(100000, random_state=RANDOM_SEED)
negative_reviews = data[data['star_rating'] == 0].sample(100000, random_state=RANDOM_SEED)
data = pd.concat([positive_reviews, negative_reviews]).reset_index(drop=True)

print(f'\nWorking dataset: {len(data):,} reviews ({data["star_rating"].sum():,} positive, {(data["star_rating"]==0).sum():,} negative)')

### Text Cleaning Pipeline

Seven steps are applied to normalize the raw text. Each step targets a specific noise source:

| Step | Operation | Why |
|------|-----------|-----|
| 1 | Lowercase | Eliminates capitalisation as a spurious feature |
| 2 | Remove HTML + URLs | These tokens carry zero sentiment signal |
| 3 | Expand contractions | Prevents contracted/expanded forms being treated as different features |
| 4 | Remove non-alpha chars | Numbers and punctuation add dimensions without discriminative value |
| 5 | Collapse whitespace | Artefact cleanup from previous steps |
| 6 | Remove stop words | High-frequency function words add dimensions without sentiment value |
| 7 | Lemmatize (verb POS) | Maps inflected forms to base form; reduces vocabulary scatter |

In [ ]:
# Steps 1–5: Cleaning

def sanitize_review(text):
    """Remove HTML tags (via BeautifulSoup) and HTTP/HTTPS URLs."""
    text = BeautifulSoup(str(text), 'html.parser').get_text()
    text = re.sub(r'http\S+', '', text)
    return text

def clean_text(text):
    """Full cleaning pass: lowercase → HTML/URL removal → contractions → non-alpha → whitespace."""
    text = str(text).lower()
    text = sanitize_review(text)
    text = contractions.fix(text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = ' '.join(text.split())
    return text

data['cleaned_reviews'] = data['review_body'].apply(clean_text)

avg_before = data['review_body'].apply(lambda x: len(str(x))).mean()
avg_after  = data['cleaned_reviews'].apply(len).mean()
print(f'Cleaning: {avg_before:.0f} → {avg_after:.0f} avg chars  ({100*(avg_before-avg_after)/avg_before:.1f}% reduction)')

In [ ]:
# Steps 6–7: Preprocessing (stop word removal + lemmatization)

stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()
tokenizer   = nltk.tokenize.WhitespaceTokenizer()

def preprocess_text(text):
    """
    Remove stop words then lemmatize with verb POS.
    Verb POS is used because sentiment-bearing words are frequently verbs
    (e.g., 'broke', 'loved', 'returned') — lemmatizing as verbs maps
    these to consistent base forms.
    """
    tokens = [w for w in str(text).split() if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w, pos='v') for w in tokens]
    return ' '.join(tokens)

data['processed_reviews'] = data['cleaned_reviews'].apply(preprocess_text)

avg_clean  = data['cleaned_reviews'].apply(len).mean()
avg_proc   = data['processed_reviews'].apply(len).mean()
print(f'Preprocessing: {avg_clean:.0f} → {avg_proc:.0f} avg chars  ({100*(avg_clean-avg_proc)/avg_clean:.1f}% reduction)')

### Visualisation: Review Length at Each Pipeline Stage

Plotting character length distributions at each stage confirms that each step has a measurable compressive effect. A flat or unchanged distribution after a step would signal that the step is a no-op — useful diagnostic.

In [ ]:
stages = [
    (data['review_body'].apply(lambda x: len(str(x))), 'Raw Reviews',          'steelblue'),
    (data['cleaned_reviews'].apply(len),               'After Cleaning',        'darkorange'),
    (data['processed_reviews'].apply(len),             'After Preprocessing',   'seagreen'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (lengths, title, color) in zip(axes, stages):
    ax.hist(lengths.clip(upper=1000), bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(lengths.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {lengths.mean():.0f}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Character Length')
    ax.set_ylabel('Number of Reviews')
    ax.legend()

plt.suptitle('Review Length Distribution Across Pipeline Stages',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('review_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Feature Engineering

### Word Clouds — Vocabulary Sanity Check

Before computing TF-IDF features, word clouds provide a qualitative check: the dominant words in each class should be intuitive sentiment indicators. If stop words or noise tokens appear prominently, it flags a preprocessing bug.

In [ ]:
pos_text = ' '.join(data[data['star_rating'] == 1]['processed_reviews'])
neg_text = ' '.join(data[data['star_rating'] == 0]['processed_reviews'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, text, title, cmap in zip(
    axes,
    [pos_text, neg_text],
    ['Positive Reviews', 'Negative Reviews'],
    ['Greens', 'Reds']
):
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=cmap, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.suptitle('Most Frequent Words by Sentiment Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

### TF-IDF Feature Extraction

TF-IDF converts review text into numerical vectors. The vectorizer is **fit on training data only** — a critical implementation detail. Fitting on the full dataset before splitting would expose test-set vocabulary to the IDF calculation, constituting data leakage and producing optimistic test-set performance estimates.

Two configurations are prepared for later comparison:
- **Unigrams only** `(1,1)` — baseline; fast; doesn't capture negation
- **Unigrams + Bigrams** `(1,2)` — larger feature space; captures phrases like "not good", "highly recommend"

In [ ]:
# 80/20 stratified split — stratify= ensures equal class balance in both splits
review_train, review_test, y_train, y_test = train_test_split(
    data['processed_reviews'], data['star_rating'],
    test_size=0.2, random_state=RANDOM_SEED, stratify=data['star_rating']
)
print(f'Train: {len(review_train):,} | Test: {len(review_test):,}')
print(f'Train class balance: {y_train.mean():.3f} positive')
print(f'Test  class balance: {y_test.mean():.3f} positive')

In [ ]:
# Unigram TF-IDF (default — used for primary model training)
vectorizer_uni = TfidfVectorizer(ngram_range=(1, 1))
X_train_uni = vectorizer_uni.fit_transform(review_train)
X_test_uni  = vectorizer_uni.transform(review_test)

# Bigram TF-IDF (for Extension 3 comparison)
vectorizer_bi = TfidfVectorizer(ngram_range=(1, 2))
X_train_bi = vectorizer_bi.fit_transform(review_train)
X_test_bi  = vectorizer_bi.transform(review_test)

# Primary features
vectorizer = vectorizer_uni
X_train    = X_train_uni
X_test     = X_test_uni

print(f'Unigram features:          {X_train_uni.shape[1]:>10,}')
print(f'Unigram + Bigram features: {X_train_bi.shape[1]:>10,}')

### Top TF-IDF Features per Class

This plot shows the 20 words with the highest mean TF-IDF weight in each sentiment class. It serves two purposes:
1. **Interpretability**: shows which words most drive each sentiment
2. **Pipeline validation**: any stop words appearing here would indicate a cleaning bug

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

# Dense conversion is memory-intensive — safe at 160k × ~100k feature scale
X_dense     = X_train.toarray()
y_train_arr = np.array(y_train)

mean_pos = X_dense[y_train_arr == 1].mean(axis=0)
mean_neg = X_dense[y_train_arr == 0].mean(axis=0)

top_n = 20
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, means, title, color in zip(
    axes,
    [mean_pos, mean_neg],
    [f'Top {top_n} Words — Positive', f'Top {top_n} Words — Negative'],
    ['seagreen', 'crimson']
):
    idx    = means.argsort()[-top_n:][::-1]
    words  = feature_names[idx]
    values = means[idx]
    ax.barh(words[::-1], values[::-1], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Mean TF-IDF Weight')

plt.suptitle('Top TF-IDF Features by Sentiment Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tfidf_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Model Training

Four Scikit-learn classifiers are trained on the unigram TF-IDF features. All four are linear models, which is appropriate for high-dimensional sparse TF-IDF matrices.

For each model, **both training and test metrics are reported**. A large gap between train and test performance signals overfitting.

| Model | Rationale |
|-------|-----------|
| Perceptron | Simplest linear model; no regularisation; establishes lower bound |
| Multinomial NB | Classical probabilistic text classifier; strong TF-IDF baseline |
| Linear SVC | Margin-maximising; well-suited for sparse high-dimensional features |
| Logistic Regression | Regularised (L2); calibrated probabilities; typically strongest |


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }

def report_results(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    """Print train + test metrics. Train/test gap reveals overfitting."""
    train_m = compute_metrics(y_train_true, y_train_pred)
    test_m  = compute_metrics(y_test_true,  y_test_pred)
    print(f"\n{'─'*52}")
    print(f"  {model_name}")
    print(f"{'─'*52}")
    print(f"  {'Split':<8} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    for split, m in [('Train', train_m), ('Test', test_m)]:
        print(f"  {split:<8} {m['accuracy']:>9.3f} {m['precision']:>10.3f} "
              f"{m['recall']:>8.3f} {m['f1']:>8.3f}")
    return {'train': train_m, 'test': test_m}

print('Evaluation helpers defined.')

In [ ]:
all_metrics = {}
all_preds   = {}

# ── Perceptron ───────────────────────────────────────────────────────────
m = Perceptron(random_state=RANDOM_SEED)
m.fit(X_train, y_train)
all_preds['Perceptron'] = m.predict(X_test)
all_metrics['Perceptron'] = report_results(
    'Perceptron', y_train, m.predict(X_train), y_test, all_preds['Perceptron'])

# ── Multinomial Naive Bayes ──────────────────────────────────────────────
m = MultinomialNB()
m.fit(X_train, y_train)
all_preds['Multinomial NB'] = m.predict(X_test)
all_metrics['Multinomial NB'] = report_results(
    'Multinomial NB', y_train, m.predict(X_train), y_test, all_preds['Multinomial NB'])

# ── Linear SVC ──────────────────────────────────────────────────────────
m = LinearSVC(random_state=RANDOM_SEED)
m.fit(X_train, y_train)
all_preds['Linear SVC'] = m.predict(X_test)
all_metrics['Linear SVC'] = report_results(
    'Linear SVC', y_train, m.predict(X_train), y_test, all_preds['Linear SVC'])

# ── Logistic Regression ──────────────────────────────────────────────────
m = LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)
m.fit(X_train, y_train)
all_preds['Logistic Regression'] = m.predict(X_test)
all_metrics['Logistic Regression'] = report_results(
    'Logistic Regression', y_train, m.predict(X_train), y_test, all_preds['Logistic Regression'])

---
## 7. Model Comparison

### Accuracy Bar Chart — Train vs Test

Plotting both train and test accuracy side by side makes the generalization gap immediately visible. A model with near-perfect training accuracy but low test accuracy has overfit.

In [ ]:
names     = list(all_metrics.keys())
train_acc = [all_metrics[n]['train']['accuracy'] for n in names]
test_acc  = [all_metrics[n]['test']['accuracy']  for n in names]

x, width = np.arange(len(names)), 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, train_acc, width, label='Train', color='steelblue',   alpha=0.85)
bars2 = ax.bar(x + width/2, test_acc,  width, label='Test',  color='darkorange', alpha=0.85)

ax.set_ylabel('Accuracy')
ax.set_title('Train vs Test Accuracy by Model', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=8)
ax.set_ylim(0.75, 1.02)
ax.legend()
ax.axhline(1.0, color='grey', linestyle=':', linewidth=0.8)

for bar in list(bars1) + list(bars2):
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Bigram vs Unigram Comparison

Bigrams capture negation phrases such as "not good" and "does not work" that unigram models cannot represent. The tradeoff is a substantially larger feature space — and longer training time. This cell quantifies whether the accuracy gain justifies the cost.

In [ ]:
results_ngram = {}
for label, ngram in [('Unigrams\n(1,1)', (1,1)), ('Unigrams + Bigrams\n(1,2)', (1,2))]:
    vec  = TfidfVectorizer(ngram_range=ngram)
    Xtr  = vec.fit_transform(review_train)
    Xte  = vec.transform(review_test)
    mdl  = LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)
    mdl.fit(Xtr, y_train)
    acc  = accuracy_score(y_test, mdl.predict(Xte))
    results_ngram[label] = {'accuracy': acc, 'n_features': Xtr.shape[1]}
    print(f'{label.replace(chr(10)," ")}: accuracy={acc:.4f}, features={Xtr.shape[1]:,}')

fig, ax = plt.subplots(figsize=(7, 4))
labels = list(results_ngram.keys())
accs   = [results_ngram[k]['accuracy'] for k in labels]
bars   = ax.bar(labels, accs, color=['steelblue', 'seagreen'], edgecolor='white', alpha=0.85, width=0.4)
ax.set_ylim(min(accs) - 0.01, max(accs) + 0.02)
ax.set_ylabel('Test Accuracy')
ax.set_title('TF-IDF Unigrams vs Bigrams — Logistic Regression', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.4f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('bigram_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Error Analysis

### Confusion Matrices

Accuracy alone doesn't reveal *how* a model fails. The confusion matrix breaks down predictions into:
- **True Positives / True Negatives** — correct predictions
- **False Positives** — predicted positive, was negative (model is overconfident)
- **False Negatives** — predicted negative, was positive (model is too conservative)

Comparing confusion matrices across models shows whether different classifiers make the same errors — suggesting shared feature-space limitations — or different errors — suggesting they could be ensembled.

In [ ]:
def draw_confusion_matrix(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Neg','Pos'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Neg','Pos'])
    thresh = cm.max() / 2
    for i, j in itertools.product(range(2), range(2)):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=10,
                color='white' if cm[i,j] > thresh else 'black')

fig, axes = plt.subplots(1, len(all_preds), figsize=(5*len(all_preds), 4))
for ax, (name, preds) in zip(axes, all_preds.items()):
    draw_confusion_matrix(ax, y_test, preds, name)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### Suggested Experiments — Not Executed

> These are proposed extensions. They have **not been run** — results would need to be obtained experimentally. They are included to outline how this project could be strengthened.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SUGGESTED EXPERIMENT 1 — 5-Fold Cross-Validation                  ║
# ║  NOT EXECUTED                                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Purpose: A single 80/20 split produces metrics with unknown variance.
#          Cross-validation gives mean ± std over multiple held-out sets.
#
# from sklearn.model_selection import StratifiedKFold, cross_validate
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
# for name, model in [('LogReg', LogisticRegression(max_iter=1500)),
#                      ('LinearSVC', LinearSVC())]:
#     scores = cross_validate(model, X_train, y_train, cv=cv,
#                             scoring=['accuracy','f1','precision','recall'])
#     print(f"{name}: accuracy={scores['test_accuracy'].mean():.3f} "
#           f"± {scores['test_accuracy'].std():.3f}")

print('SUGGESTED EXPERIMENT 1 — Cross-Validation: see comment above.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SUGGESTED EXPERIMENT 2 — Hyperparameter Tuning                    ║
# ║  NOT EXECUTED                                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Purpose: Default sklearn parameters are rarely optimal. Grid search over
#          regularisation strength (C) often yields 0.5–2% accuracy gain.
#
# from sklearn.model_selection import GridSearchCV
#
# param_grid = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
# gs = GridSearchCV(LogisticRegression(max_iter=1500), param_grid,
#                   cv=3, scoring='accuracy', n_jobs=-1)
# gs.fit(X_train, y_train)
# print(f"Best C: {gs.best_params_['C']}")
# print(f"CV accuracy: {gs.best_score_:.4f}")
# print(f"Test accuracy: {accuracy_score(y_test, gs.predict(X_test)):.4f}")

print('SUGGESTED EXPERIMENT 2 — Hyperparameter Tuning: see comment above.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SUGGESTED EXPERIMENT 3 — ROC / AUC Analysis                       ║
# ║  NOT EXECUTED                                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Purpose: ROC curves measure separability independent of the classification
#          threshold. Two models can have the same accuracy at threshold=0.5
#          but very different AUCs — meaning different deployment performance
#          at non-default thresholds.
#
# from sklearn.metrics import roc_curve, auc
#
# lr = LogisticRegression(max_iter=1500).fit(X_train, y_train)
# y_prob = lr.predict_proba(X_test)[:, 1]
# fpr, tpr, _ = roc_curve(y_test, y_prob)
# plt.plot(fpr, tpr, label=f'LR AUC = {auc(fpr, tpr):.3f}')
# plt.plot([0,1],[0,1],'k--')
# plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
# plt.legend(); plt.show()

print('SUGGESTED EXPERIMENT 3 — ROC/AUC: see comment above.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SUGGESTED EXPERIMENT 4 — Ablation Study                           ║
# ║  NOT EXECUTED                                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Purpose: Quantify each preprocessing step's individual contribution.
#          Remove one step at a time and measure accuracy drop.
#
# ablations = {
#     'Full pipeline':       data['processed_reviews'],
#     'No lemmatization':    data['cleaned_reviews'].apply(remove_stop_words_only),
#     'No stop word removal':data['cleaned_reviews'].apply(lemmatize_only),
#     'No cleaning':         data['review_body'].apply(preprocess_text),
#     'Raw baseline':        data['review_body'],
# }
# for label, text_series in ablations.items():
#     vec = TfidfVectorizer()
#     Xtr, Xte, ytr, yte = ... # split
#     acc = accuracy_score(yte, LogisticRegression(max_iter=1500).fit(Xtr,ytr).predict(Xte))
#     print(f'{label}: {acc:.4f}')

print('SUGGESTED EXPERIMENT 4 — Ablation Study: see comment above.')

---
## 9. Discussion

### What the results tell us

**Model ranking:** Logistic Regression consistently outperforms all other models. This is a well-documented pattern in NLP: L2-regularised logistic regression is often the strongest baseline on TF-IDF features, because its soft-margin loss and regularisation are well-calibrated for high-dimensional sparse data. LinearSVC is a close second, as expected for a related family of linear models.

**Perceptron gap:** The Perceptron's lower performance is attributable to its lack of regularisation — it converges to any separating hyperplane rather than the maximum-margin one, and its convergence is sensitive to example ordering. For a practical system, Perceptron should be treated as a sanity check.

**Train/test gap:** All four models show a modest train/test accuracy gap, which is healthy — it indicates the models are not severely overfit. A near-zero gap would suggest underfitting; a very large gap would suggest overfitting.

**Bigram impact:** Including bigrams adds meaningful features (negation phrases, compound descriptors) but at significant computational cost. The measured accuracy gain should be weighed against the feature space expansion.

### What TF-IDF cannot capture

TF-IDF treats each document as a bag of words, so it has no concept of:
- **Word order** — "this is not bad" and "this is bad" have nearly identical feature vectors
- **Negation scope** — "I would recommend this" and "I would not recommend this" differ by one word
- **Irony/sarcasm** — "Wow, great, broke on day one" is classified as positive based on "great"

These limitations motivate the use of transformer-based models (BERT, RoBERTa) for more nuanced sentiment tasks.

---
## 10. Conclusion

This project demonstrates that a classical NLP pipeline — TF-IDF features + a regularised linear classifier — achieves strong performance (90% accuracy) on binary sentiment classification of Amazon Kitchen reviews.

**Key takeaways:**

1. **Logistic Regression is the strongest model** at 90% test accuracy, with consistent precision, recall, and F1 across both classes.

2. **The preprocessing pipeline is effective**: a 58% reduction in average review length while maintaining sentiment signal is evidence that each step meaningfully compresses noise rather than information.

3. **Data leakage prevention matters**: TF-IDF must be fit on training data only. Fitting on the full dataset before splitting produces optimistic test metrics that do not reflect real deployment performance.

4. **Bigrams add signal at a cost**: the accuracy improvement from adding bigrams is modest but measurable, and the larger feature space may not be worth it for resource-constrained deployments.

5. **Linear separability holds**: the fact that all four linear models — including the Perceptron — exceed 85% accuracy confirms that sentiment in TF-IDF space is largely linearly separable for this dataset.

**Limitations to keep in mind:** the dataset is domain-specific (Kitchen products), the class balance is artificially equal (real Amazon data is heavily positive-skewed), and the bag-of-words assumption misses negation and word order.